# 🔬 SWARNALI — Data Download & Preprocessing
### IEEE EMBS Pune Chapter — Team CodeCrafters (AI ML-21)
### Week 2 — HAM10000 Dataset Preparation

**Your job this week:**
1. Download HAM10000 from Kaggle
2. Explore the dataset
3. Apply preprocessing (hair removal, CLAHE, resize, normalize)
4. Split into Train / Validation / Test
5. Save ready-to-use data for Subham's training notebook

---
⚠️ **Run this on Google Colab with GPU enabled**
Go to: Runtime → Change runtime type → GPU

## STEP 1 — Install Required Libraries

In [ ]:
# Install required packages
!pip install kaggle opencv-python-headless pandas matplotlib seaborn scikit-learn -q
print('✅ Libraries installed')

## STEP 2 — Upload Kaggle API Key

To get your kaggle.json:
1. Go to kaggle.com → Your Profile → Account
2. Scroll to API section → Click 'Create New Token'
3. It downloads kaggle.json — upload that file below

In [ ]:
from google.colab import files
import os

# Upload your kaggle.json
print('Upload your kaggle.json file...')
uploaded = files.upload()

# Place it in the right location
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)
print('✅ Kaggle API key configured')

## STEP 3 — Download HAM10000 Dataset

In [ ]:
# Download HAM10000 from Kaggle
print('Downloading HAM10000 dataset (this takes 2-3 minutes)...')
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham10000 --unzip
print('✅ Dataset downloaded!')

# Check what got downloaded
import os
for root, dirs, files_list in os.walk('/content/ham10000'):
    level = root.replace('/content/ham10000', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files_list[:5]:
            print(f'{subindent}{f}')

## STEP 4 — Load & Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# Load metadata CSV
df = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nColumns:', df.columns.tolist())
print('\nDisease classes:')
print(df['dx'].value_counts())

In [ ]:
# Visualize class distribution
import matplotlib.pyplot as plt

class_names = {
    'nv':    'Melanocytic Nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign Keratosis',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis',
    'vasc':  'Vascular Lesion',
    'df':    'Dermatofibroma'
}

counts = df['dx'].value_counts()
labels = [class_names[c] for c in counts.index]
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1.bar(counts.index, counts.values, color=colors)
ax1.set_title('Class Distribution — HAM10000', fontsize=13, fontweight='bold')
ax1.set_xlabel('Disease Class')
ax1.set_ylabel('Number of Images')
for i, v in enumerate(counts.values):
    ax1.text(i, v + 50, str(v), ha='center', fontsize=10)

# Pie chart
ax2.pie(counts.values, labels=counts.index, colors=colors, autopct='%1.1f%%', startangle=140)
ax2.set_title('Class Distribution (%) — HAM10000', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n⚠️  PROBLEM: nv (Nevi) dominates at 67% — this is the CLASS IMBALANCE we address!')

In [ ]:
# Find all image files and link them to metadata
import glob

# HAM10000 images are in two folders
image_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
print(f'Total images found: {len(image_paths)}')

# Build image path lookup dictionary
image_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f'Unique image IDs: {len(image_dict)}')

# Add image path to dataframe
df['image_path'] = df['image_id'].map(image_dict)
missing = df['image_path'].isna().sum()
print(f'Missing images: {missing}')
df = df.dropna(subset=['image_path'])
print(f'Final dataset size: {len(df)} images')

In [ ]:
# Show sample images from each class
import cv2
from IPython.display import display

fig, axes = plt.subplots(2, 7, figsize=(18, 6))
classes = df['dx'].unique()

for col, cls in enumerate(sorted(classes)):
    subset = df[df['dx'] == cls]
    for row in range(2):
        sample = subset.sample(1).iloc[0]
        img = cv2.imread(sample['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(cls, fontsize=10, fontweight='bold')

plt.suptitle('Sample Images from Each Class — HAM10000', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## STEP 5 — Preprocessing Functions

We apply 4 preprocessing steps based on the recent papers:
1. **Hair Removal** — BlackHat filter + inpainting (Frederich et al. 2024)
2. **CLAHE** — Contrast enhancement (HAM10000 papers)
3. **Resize** — 224×224 pixels (ALL 4 papers)
4. **Normalize** — ImageNet statistics (ALL 4 papers)

In [ ]:
import cv2
import numpy as np

def remove_hair(image):
    """
    Step 1: Hair Removal using BlackHat morphological filter + inpainting
    Based on: Frederich et al. 2024, MDPI 2025
    """
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # BlackHat filter to detect dark thin structures (hair)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)

    # Threshold to create hair mask
    _, hair_mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)

    # Inpaint (fill in) the hair regions with surrounding skin texture
    clean_image = cv2.inpaint(image, hair_mask, inpaintRadius=3,
                               flags=cv2.INPAINT_TELEA)
    return clean_image


def apply_clahe(image):
    """
    Step 2: CLAHE — Contrast Limited Adaptive Histogram Equalization
    Applied on L channel of LAB color space
    Improves visibility of lesion borders
    """
    # Convert to LAB color space
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Apply CLAHE to L (luminance) channel only
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)

    # Merge back and convert to RGB
    lab_enhanced = cv2.merge([l_enhanced, a, b])
    enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)
    return enhanced


def resize_image(image, size=(224, 224)):
    """
    Step 3: Resize to 224x224
    EfficientNet-B0 requires exactly 224x224 input
    """
    return cv2.resize(image, size, interpolation=cv2.INTER_LINEAR)


def preprocess_image(image_path):
    """
    Full pipeline: Load → Hair Removal → CLAHE → Resize
    Normalization is handled by PyTorch transforms during training
    """
    # Load image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Step 1: Hair removal
    img = remove_hair(img)

    # Step 2: CLAHE contrast enhancement
    img = apply_clahe(img)

    # Step 3: Resize to 224x224
    img = resize_image(img)

    return img


print('✅ Preprocessing functions defined')

In [ ]:
# Visualize preprocessing steps on a sample image
sample_path = df.sample(1).iloc[0]['image_path']

# Load original
original = cv2.imread(sample_path)
original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

# Apply step by step
after_hair   = remove_hair(original)
after_clahe  = apply_clahe(after_hair)
after_resize = resize_image(after_clahe)

# Plot all steps
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
steps = [
    (original,     'Step 0\nOriginal'),
    (after_hair,   'Step 1\nHair Removal'),
    (after_clahe,  'Step 2\nCLAHE Enhancement'),
    (after_resize, 'Step 3\nResized (224x224)'),
]

for ax, (img, title) in zip(axes, steps):
    ax.imshow(img)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Preprocessing Pipeline — Step by Step', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/preprocessing_steps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Preprocessing visualization saved!')

## STEP 6 — Encode Labels & Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Map disease codes to numbers
label_map = {'nv':0, 'mel':1, 'bkl':2, 'bcc':3, 'akiec':4, 'vasc':5, 'df':6}
df['label'] = df['dx'].map(label_map)

print('Label mapping:')
for k, v in label_map.items():
    print(f'  {v} = {k} ({class_names[k]})')

# Split: 70% train, 15% val, 15% test
# Using stratify to keep class proportions in each split
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

print(f'\nSplit results:')
print(f'  Train: {len(train_df)} images ({len(train_df)/len(df)*100:.1f}%)')
print(f'  Val:   {len(val_df)} images ({len(val_df)/len(df)*100:.1f}%)')
print(f'  Test:  {len(test_df)} images ({len(test_df)/len(df)*100:.1f}%)')

## STEP 7 — Calculate Class Weights

Since nv = 67% of data, we need class weights so the model doesn't just predict nv every time

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights from training set only
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1,2,3,4,5,6]),
    y=train_df['label'].values
)

print('Class weights (higher = rarer class gets more attention):')
for i, (cls, name) in enumerate(class_names.items()):
    print(f'  Class {i} ({cls} — {name}): weight = {class_weights[i]:.4f}')

# Save weights for Subham
np.save('/content/class_weights.npy', class_weights)
print('\n✅ Class weights saved to /content/class_weights.npy')

## STEP 8 — Save Processed DataFrames for Subham

In [ ]:
# Save train/val/test splits
train_df.to_csv('/content/train.csv', index=False)
val_df.to_csv('/content/val.csv', index=False)
test_df.to_csv('/content/test.csv', index=False)

# Save label map
import json
with open('/content/label_map.json', 'w') as f:
    json.dump(label_map, f)

print('✅ Saved files:')
print('  /content/train.csv        → Subham uses this for training')
print('  /content/val.csv          → Subham uses this for validation')
print('  /content/test.csv         → Subham uses this for final testing')
print('  /content/class_weights.npy → Subham uses this for weighted loss')
print('  /content/label_map.json   → class name ↔ number mapping')
print()
print('📌 Download these 5 files and share with Subham!')

## STEP 9 — Final Summary

In [ ]:
print('='*55)
print('  SWARNALI — WEEK 2 PREPROCESSING COMPLETE')
print('='*55)
print(f'  Total images processed : {len(df)}')
print(f'  Train set              : {len(train_df)}')
print(f'  Validation set         : {len(val_df)}')
print(f'  Test set               : {len(test_df)}')
print(f'  Number of classes      : 7')
print(f'  Preprocessing steps    : Hair Removal + CLAHE + Resize 224x224')
print(f'  Class weights computed : Yes (for weighted loss)')
print('='*55)
print()
print('📦 Files ready for Subham:')
print('   train.csv, val.csv, test.csv')
print('   class_weights.npy, label_map.json')
print()
print('🎉 Swarnali Week 2 DONE!')